# Session 5 · Part 2 — Foundation Models for Earth Observation
## 🎓 STUDENT VERSION

**AI for Environmental Sustainability** · Dr. René Traoré, Eva Vormschlag, Prof. Marius Lindauer  
📦 Repository: https://github.com/automl-edu/AI_Environmental_Sustainability

---

## Learning objectives

1. Train a small CNN from scratch on EuroSAT and record baseline accuracy
2. Fine-tune a pretrained Geospatial Foundation Model (GeoFM) and compare
3. Plot a **data-efficiency curve** (100% → 5% labels)
4. Analyse a confusion matrix at very low label regimes

**Estimated time:** ~25 minutes  
**Default GeoFM:** TorchGeo pretrained ResNet50 (Sentinel-2, MoCo self-supervised)


## Section 0 · Environment Setup

In [ ]:
import os, sys

def get_runtime():
    if 'google.colab' in sys.modules:
        return 'colab'
    elif 'SLURM_JOB_ID' in os.environ or 'KISSKI' in os.environ.get('HPC_NAME', ''):
        return 'hpc'
    else:
        return 'local'

RUNTIME = get_runtime()
print(f"Runtime detected: {RUNTIME}")

if RUNTIME == 'colab':
    try:
        from google.colab import drive
        print("Attempting to mount Google Drive...")
        drive.mount('/content/drive')
        DATA_ROOT      = '/content/drive/MyDrive/AI4ES_Seminar/data'
        CHECKPOINT_DIR = '/content/drive/MyDrive/AI4ES_Seminar/checkpoints'
        DOWNLOAD       = False
    except Exception as e:
        print(f"Google Drive mount failed: {e}")
        print("Falling back to local Colab storage...")
        DATA_ROOT      = '/content/data'
        CHECKPOINT_DIR = '/content/checkpoints'
        os.makedirs(DATA_ROOT, exist_ok=True)
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        DOWNLOAD       = True
elif RUNTIME == 'hpc':
    # On KISSKI, $HOME is on vast-kisski (~120 GB) — use it for data and checkpoints
    # /opt/conda is a separate system partition that is typically full — never write there
    DATA_ROOT      = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'data')
    CHECKPOINT_DIR = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'checkpoints')
    os.makedirs(DATA_ROOT,      exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True
    local_packages = os.path.expanduser('~/.local/lib/python3.12/site-packages')
    if local_packages not in sys.path:
        sys.path.insert(0, local_packages)
else:
    DATA_ROOT      = os.path.expanduser('~/AI4ES_Seminar/data')
    CHECKPOINT_DIR = os.path.expanduser('~/AI4ES_Seminar/checkpoints')
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True

print(f"Data root: {DATA_ROOT}")


In [ ]:
import subprocess

packages = [
    'torch>=2.0.0', 'torchvision>=0.15.0', 'torchgeo>=0.6.0',
    'transformers>=4.35.0', 'timm>=0.9.0',
    'numpy>=1.24.0', 'matplotlib>=3.7.0',
    'scikit-learn>=1.3.0', 'seaborn>=0.12.0'
]

try:
    import torch, torchgeo, transformers, timm, sklearn, seaborn
    print(f'All packages already available (torch {torch.__version__}, torchgeo {torchgeo.__version__})')
except ImportError:
    pip_tmp = os.path.expanduser('~/.local/pip_tmp')
    os.makedirs(pip_tmp, exist_ok=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--user', '--no-cache-dir'] + packages,
        check=True,
        env={**os.environ, 'TMPDIR': pip_tmp}
    )
    print('Packages installed into ~/.local (--user mode)')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.models as tv_models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
from torchgeo.datasets import EuroSAT
from torchgeo.models import ResNet50_Weights
import random, time, copy, warnings

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*encountered in matmul")

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Device: {DEVICE}")

# macOS/Windows use 'spawn' for DataLoader workers — can't pickle notebook-defined
# collate_fn. Use 0 workers on non-Linux to avoid worker crashes.
NUM_WORKERS     = 2 if sys.platform.startswith('linux') else 0
NUM_CLASSES     = 10
IN_CHANNELS     = 13
BATCH_SIZE      = 64
EPOCHS_SCRATCH  = 10
EPOCHS_FINETUNE = 5
LR              = 1e-3
LABEL_FRACTIONS = [1.0, 0.5, 0.25, 0.1, 0.05]
print(f"NUM_WORKERS: {NUM_WORKERS}")


## Section 1 · Data Loading

In [ ]:
train_dataset = EuroSAT(root=DATA_ROOT, split='train', bands=EuroSAT.all_band_names, download=True)
val_dataset   = EuroSAT(root=DATA_ROOT, split='val',   bands=EuroSAT.all_band_names, download=True)
test_dataset  = EuroSAT(root=DATA_ROOT, split='test',  bands=EuroSAT.all_band_names, download=True)

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}  |  Test: {len(test_dataset)}")

def collate_fn(batch):
    images = torch.stack([s['image'].float() for s in batch])
    labels = torch.tensor([s['label'] for s in batch])
    return images, labels

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=NUM_WORKERS)


In [ ]:
def get_label_subset(dataset, fraction, seed=SEED):
    rng = np.random.default_rng(seed)
    n   = int(len(dataset) * fraction)
    idx = rng.choice(len(dataset), size=n, replace=False).tolist()
    return Subset(dataset, idx)


## Section 2 · Training Infrastructure

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
    return accuracy_score(all_labels, all_preds), all_preds, all_labels

def run_training(model, train_loader, val_loader, epochs, lr=LR, label='model'):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_acc, _, _         = evaluate(model, val_loader)
        scheduler.step()
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())
        print(f"[{label}] Epoch {epoch:02d}/{epochs}  "
              f"loss={train_loss:.4f}  train={train_acc:.3f}  val={val_acc:.3f}  ({time.time()-t0:.1f}s)")
    model.load_state_dict(best_state)
    return best_val_acc, model


## Section 3 · CNN Trained from Scratch

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))


In [ ]:
# ── TODO: train CNN from scratch with 100% labels ────────────────────────────
# 1. Build a DataLoader from train_dataset with BATCH_SIZE, shuffle=True,
#    collate_fn=collate_fn, num_workers=NUM_WORKERS
# 2. Instantiate SmallCNN and move to DEVICE
# 3. Call run_training for EPOCHS_SCRATCH epochs
# 4. Store the best val accuracy in cnn_val_acc
raise NotImplementedError


## Section 4 · Pretrained GeoFM Fine-tuning

In [ ]:
def build_geofm(num_classes=NUM_CLASSES, freeze_backbone=False):
    weights  = ResNet50_Weights.SENTINEL2_ALL_MOCO
    backbone = tv_models.resnet50(weights=None)
    backbone.conv1 = nn.Conv2d(IN_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
    state_dict = weights.get_state_dict(progress=True)
    msg = backbone.load_state_dict(state_dict, strict=False)
    print(f"Pretrained weights loaded. Missing: {len(msg.missing_keys)}")
    if freeze_backbone:
        for p in backbone.parameters():
            p.requires_grad = False
    backbone.fc = nn.Linear(backbone.fc.in_features, num_classes)
    return backbone


In [ ]:
# ── TODO: fine-tune GeoFM with 100% labels ───────────────────────────────────
# 1. Call build_geofm(freeze_backbone=False) and move to DEVICE
# 2. Run run_training for EPOCHS_FINETUNE epochs with lr=5e-4
# 3. Store best val accuracy in geofm_val_acc_100
# 4. Print comparison with cnn_val_acc
raise NotImplementedError


## Section 5 · Data-Efficiency Curve

In [ ]:
# ── TODO: label-efficiency loop ───────────────────────────────────────────────
# For each fraction in LABEL_FRACTIONS:
#   1. Build a subset with get_label_subset
#   2. Build a DataLoader with num_workers=NUM_WORKERS
#   3. Reinitialise a fresh GeoFM (never reuse weights between fractions)
#   4. Run run_training and store the val accuracy in results_geofm[frac]
results_geofm = {}
raise NotImplementedError


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
fracs   = sorted(results_geofm)
ax.plot([f*100 for f in fracs], [results_geofm[f] for f in fracs],
        'o-', color='steelblue', linewidth=2, markersize=7, label='GeoFM (pretrained)')
ax.axhline(cnn_val_acc, linestyle='--', color='tomato', linewidth=1.5,
           label='CNN from scratch (100% labels)')
ax.set_xlabel('Training labels used (%)'); ax.set_ylabel('Validation accuracy')
ax.set_title('Label-efficiency curve'); ax.set_xscale('log')
ax.set_xticks([5,10,25,50,100]); ax.set_xticklabels(['5%','10%','25%','50%','100%'])
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## Section 6 · Confusion Matrix at 5% Labels

In [ ]:
# ── TODO: confusion matrix at 5% labels ──────────────────────────────────────
# 1. Build a subset with get_label_subset(train_dataset, 0.05)
# 2. Train a fresh GeoFM for EPOCHS_FINETUNE epochs
# 3. Evaluate on test_loader and build a confusion_matrix
# 4. Plot with seaborn heatmap and print per-class accuracy
raise NotImplementedError


## Section 7 · Optional Extension — Linear Probing

In [ ]:
# ── OPTIONAL: Linear probing — freeze backbone, train head only ───────────────
# Shows how much of the classification signal comes from the pretrained
# representation alone, without any fine-tuning of the backbone.

geofm_frozen = build_geofm(freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in geofm_frozen.parameters() if p.requires_grad)
total     = sum(p.numel() for p in geofm_frozen.parameters())
print(f"Trainable: {trainable:,} / {total:,} parameters")

lp_val_acc, geofm_frozen = run_training(
    geofm_frozen, full_train_loader, val_loader,
    epochs=EPOCHS_FINETUNE, lr=1e-2, label='GeoFM-LinearProbe'
)
print(f"Linear probe : {lp_val_acc:.4f}")
print(f"Full finetune: {geofm_val_acc_100:.4f}")
print(f"CNN scratch  : {cnn_val_acc:.4f}")


## Section 8 · Optional Extension — Prithvi-100M (KISSKI)

In [ ]:
# ── OPTIONAL: Prithvi-100M via TerraTorch (KISSKI recommended) ───────────────
# Pre-requisite: install terratorch before the session from the login node:
#   TMPDIR=~/.local/pip_tmp pip install --user --no-cache-dir terratorch
#
# from terratorch.registry import BACKBONE_REGISTRY
# import torch, torch.nn as nn
#
# prithvi_backbone = BACKBONE_REGISTRY.build("prithvi_eo_v2_100_tl", pretrained=True)
#
# class PrithviClassifier(nn.Module):
#     def __init__(self, backbone, num_classes=NUM_CLASSES):
#         super().__init__()
#         self.backbone = backbone
#         # Prithvi-100M pretrained on 6 S2 bands: B02,B03,B04,B05,B06,B07
#         # → tensor indices [1,2,3,4,5,6] in the 13-band EuroSAT tensor
#         self.band_indices = torch.tensor([1, 2, 3, 4, 5, 6])
#         self.head = nn.Linear(backbone.embed_dim, num_classes)
#
#     def forward(self, x):
#         x = x[:, self.band_indices, :, :]   # [B, 6, H, W]
#         x = x.unsqueeze(2)                  # [B, 6, 1, H, W]
#         tokens = self.backbone(x)
#         if isinstance(tokens, (list, tuple)): tokens = tokens[0]
#         return self.head(tokens.mean(dim=1))
#
# prithvi_clf = PrithviClassifier(prithvi_backbone).to(DEVICE)
# for p in prithvi_clf.backbone.parameters(): p.requires_grad = False
# prithvi_val_acc, prithvi_clf = run_training(
#     prithvi_clf, full_train_loader, val_loader,
#     epochs=EPOCHS_FINETUNE, lr=1e-3, label='Prithvi-100M'
# )


## Discussion

1. At what label fraction does GeoFM first match CNN-100%?
2. Which two classes are most confused at 5% labels?
3. What does the linear probing result tell you about the pretrained features?
4. MoCo (contrastive) vs Prithvi (masked autoencoding) — which would you expect to produce better land-cover features?

